<a href="https://colab.research.google.com/github/hayatkhan20/umd-urban-heat-exposure/blob/main/notebooks/03_overture_building_heights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip -q install -U \
    overturemaps \
    geopandas \
    pyogrio \
    folium \
    mapclassify \
    scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 29.9 MB/s eta 0:00:00


In [25]:
import os

REPO_DIRECTORY = "/content/umd-urban-heat-exposure"

if not os.path.exists(REPO_DIRECTORY):
    !git clone https://github.com/hayatkhan20/umd-urban-heat-exposure.git
else:
    print("Repository already exists.")

%cd /content/umd-urban-heat-exposure

Repository already exists.
/content/umd-urban-heat-exposure


In [26]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd

from overturemaps.core import geodataframe, get_latest_release

BOUNDARY_PATH = "data/umd_boundary_final.geojson"

umd_boundary = (
    gpd.read_file(BOUNDARY_PATH)
    .to_crs("EPSG:4326")
)

print("Boundary features:", len(umd_boundary))
print("Geometry:", umd_boundary.geometry.iloc[0].geom_type)
print("CRS:", umd_boundary.crs)
print("Valid:", umd_boundary.geometry.is_valid.all())

umd_boundary

Boundary features: 1
Geometry: Polygon
CRS: EPSG:4326
Valid: True


,name,source,method,concave_hull_ratio,buffer_m,geometry
0,UMD College Park Analysis Area,Maryland Open Data: UMD Administrative Boundary,Concave hull with 100 m outward buffer,0.3,100,"POLYGON ((-76.96684 39.00239, -76.96694 39.002..."


In [27]:
# Bounding box order:
# west, south, east, north

west, south, east, north = (
    umd_boundary.total_bounds
)

bbox = (
    float(west),
    float(south),
    float(east),
    float(north)
)

OVERTURE_RELEASE = "2026-08-19.0"

print("Overture release:", OVERTURE_RELEASE)
print("Bounding box:", bbox)

# Download buildings intersecting the AOI bounding box
buildings_raw = geodataframe(
    "building",
    bbox=bbox,
    release=OVERTURE_RELEASE,
    stac=True
)

if buildings_raw.crs is None:
    buildings_raw = buildings_raw.set_crs(
        "EPSG:4326"
    )
else:
    buildings_raw = buildings_raw.to_crs(
        "EPSG:4326"
    )

print("Buildings inside bounding box:", len(buildings_raw))

# Clip accurately to UMD boundary
buildings = gpd.clip(
    buildings_raw,
    umd_boundary,
    keep_geom_type=True
).reset_index(drop=True)

# Remove empty or invalid records
buildings = buildings[
    buildings.geometry.notna()
    & ~buildings.geometry.is_empty
].copy()

buildings = buildings[
    buildings.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )
].reset_index(drop=True)

print("Buildings after boundary clipping:", len(buildings))
print("Geometry valid:", buildings.geometry.is_valid.all())
print("Columns:")
print(buildings.columns.tolist())

Overture release: 2026-08-19.0
Bounding box: (-76.96693516618285, 38.96627217534857, -76.91727691799831, 39.00843195106727)
Buildings inside bounding box: 9912
Buildings after boundary clipping: 3694
Geometry valid: True
Columns:
['id', 'names', 'sources', 'level', 'height', 'min_height', 'is_underground', 'num_floors', 'num_floors_underground', 'min_floor', 'subtype', 'class', 'facade_color', 'facade_material', 'roof_material', 'roof_shape', 'roof_direction', 'roof_orientation', 'roof_color', 'roof_height', 'geometry', 'has_parts', 'version', 'bbox']


In [ ]:
building_map = buildings.explore(
    color="#f47f4f",
    tooltip=[
        column for column in [
            "id",
            "height",
            "num_floors",
            "has_parts"
        ]
        if column in buildings.columns
    ],
    tiles="CartoDB positron",
    style_kwds={
        "fillOpacity": 0.70,
        "weight": 0.3
    },
    name="Overture buildings"
)

umd_boundary.explore(
    m=building_map,
    color="red",
    fill=False,
    style_kwds={"weight": 3},
    name="UMD boundary"
)

building_map

In [29]:
print("Building variable: buildings")
print("Total buildings:", len(buildings))

print("\nAvailable columns:")
print(buildings.columns.tolist())

for column in [
    "height",
    "num_floors",
    "has_parts",
    "roof_height"
]:
    if column not in buildings.columns:
        buildings[column] = np.nan

buildings["height"] = pd.to_numeric(
    buildings["height"],
    errors="coerce"
)

buildings["num_floors"] = pd.to_numeric(
    buildings["num_floors"],
    errors="coerce"
)

valid_height = buildings["height"].where(
    buildings["height"] > 0
)

valid_floors = buildings["num_floors"].where(
    buildings["num_floors"] > 0
)

height_count = valid_height.notna().sum()
floor_count = valid_floors.notna().sum()

parts_count = (
    buildings["has_parts"]
    .fillna(False)
    .astype(bool)
    .sum()
)

print("\n--- OVERTURE HEIGHT COVERAGE ---")

print(
    f"Buildings with direct height: "
    f"{height_count:,}"
)

print(
    f"Direct height coverage: "
    f"{height_count / len(buildings) * 100:.2f}%"
)

print(
    f"\nBuildings with number of floors: "
    f"{floor_count:,}"
)

print(
    f"Floor coverage: "
    f"{floor_count / len(buildings) * 100:.2f}%"
)

print(
    f"\nBuildings with detailed parts: "
    f"{parts_count:,}"
)

print(
    f"Building-part coverage: "
    f"{parts_count / len(buildings) * 100:.2f}%"
)

print("\nDirect height statistics:")
print(
    valid_height.describe(
        percentiles=[0.25, 0.50, 0.75, 0.95]
    )
)

Building variable: buildings
Total buildings: 3694

Available columns:
['id', 'names', 'sources', 'level', 'height', 'min_height', 'is_underground', 'num_floors', 'num_floors_underground', 'min_floor', 'subtype', 'class', 'facade_color', 'facade_material', 'roof_material', 'roof_shape', 'roof_direction', 'roof_orientation', 'roof_color', 'roof_height', 'geometry', 'has_parts', 'version', 'bbox']

--- OVERTURE HEIGHT COVERAGE ---
Buildings with direct height: 3,485
Direct height coverage: 94.34%

Buildings with number of floors: 177
Floor coverage: 4.79%

Buildings with detailed parts: 8
Building-part coverage: 0.22%

Direct height statistics:
count    3485.000000
mean        6.052649
std         3.902381
min         0.056311
25%         3.385150
50%         5.608388
75%         7.635028
95%        11.169723
max        69.510000
Name: height, dtype: float64


In [30]:
# UTM Zone 18N for accurate metre measurements
buildings_height = (
    buildings
    .to_crs("EPSG:26918")
    .copy()
)

buildings_height["overture_height_m"] = (
    pd.to_numeric(
        buildings_height["height"],
        errors="coerce"
    )
)

buildings_height["num_floors"] = (
    pd.to_numeric(
        buildings_height["num_floors"],
        errors="coerce"
    )
)

buildings_height["footprint_area_m2"] = (
    buildings_height.geometry.area
)

# Physically plausible limits
valid_direct_height = (
    buildings_height["overture_height_m"]
    .between(1.5, 100)
)

valid_floors = (
    buildings_height["num_floors"]
    .between(1, 30)
)

buildings_height["analysis_height_m"] = np.where(
    valid_direct_height,
    buildings_height["overture_height_m"],
    np.nan
)

buildings_height["height_source"] = np.where(
    valid_direct_height,
    "overture_height",
    "missing"
)

# Calibrate floor-to-floor height
overlap = valid_direct_height & valid_floors

floor_height_ratios = (
    buildings_height.loc[
        overlap,
        "overture_height_m"
    ]
    / buildings_height.loc[
        overlap,
        "num_floors"
    ]
)

floor_height_ratios = floor_height_ratios[
    floor_height_ratios.between(2.4, 5.0)
]

if len(floor_height_ratios) >= 10:
    FLOOR_HEIGHT_M = float(
        floor_height_ratios.median()
    )
else:
    FLOOR_HEIGHT_M = 3.2

print(
    f"Calibrated floor-to-floor height: "
    f"{FLOOR_HEIGHT_M:.2f} m"
)

print(
    "Buildings used for calibration:",
    len(floor_height_ratios)
)

floor_fallback = (
    buildings_height["analysis_height_m"].isna()
    & valid_floors
)

buildings_height.loc[
    floor_fallback,
    "analysis_height_m"
] = (
    buildings_height.loc[
        floor_fallback,
        "num_floors"
    ]
    * FLOOR_HEIGHT_M
)

buildings_height.loc[
    floor_fallback,
    "height_source"
] = "estimated_from_floors"

usable = buildings_height[
    "analysis_height_m"
].notna()

count_coverage = usable.mean() * 100

area_coverage = (
    buildings_height.loc[
        usable,
        "footprint_area_m2"
    ].sum()
    / buildings_height[
        "footprint_area_m2"
    ].sum()
    * 100
)

print("\n--- INITIAL HEIGHT COVERAGE ---")
print("Total buildings:", len(buildings_height))
print("Usable heights:", usable.sum())
print("Remaining missing:", (~usable).sum())
print(f"Count coverage: {count_coverage:.2f}%")
print(f"Area coverage: {area_coverage:.2f}%")

print("\nHeight sources:")
print(
    buildings_height[
        "height_source"
    ].value_counts()
)

Calibrated floor-to-floor height: 3.67 m
Buildings used for calibration: 92

--- INITIAL HEIGHT COVERAGE ---
Total buildings: 3694
Usable heights: 3479
Remaining missing: 215
Count coverage: 94.18%
Area coverage: 88.73%

Height sources:
height_source
overture_height          3460
missing                   215
estimated_from_floors      19
Name: count, dtype: int64


In [31]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

model_buildings = buildings_height.copy()

model_buildings["perimeter_m"] = (
    model_buildings.geometry.length
)

centroids = model_buildings.geometry.centroid

model_buildings["centroid_x"] = centroids.x
model_buildings["centroid_y"] = centroids.y

model_buildings["log_area"] = np.log1p(
    model_buildings["footprint_area_m2"]
)

model_buildings["log_perimeter"] = np.log1p(
    model_buildings["perimeter_m"]
)

model_buildings["compactness"] = (
    4
    * np.pi
    * model_buildings["footprint_area_m2"]
    / model_buildings["perimeter_m"].pow(2)
)

bounds = model_buildings.geometry.bounds

bbox_area = (
    (bounds["maxx"] - bounds["minx"])
    * (bounds["maxy"] - bounds["miny"])
)

model_buildings["bbox_fill_ratio"] = (
    model_buildings["footprint_area_m2"]
    / bbox_area.replace(0, np.nan)
)

model_buildings["floors_feature"] = (
    pd.to_numeric(
        model_buildings["num_floors"],
        errors="coerce"
    ).fillna(0)
)

model_buildings["floors_known"] = (
    model_buildings["floors_feature"] > 0
).astype(int)

model_buildings["roof_height_feature"] = (
    pd.to_numeric(
        model_buildings["roof_height"],
        errors="coerce"
    ).fillna(0)
)

model_buildings["roof_height_known"] = (
    model_buildings["roof_height_feature"] > 0
).astype(int)

model_buildings["has_parts_feature"] = (
    model_buildings["has_parts"]
    .fillna(False)
    .astype(bool)
    .astype(int)
)

feature_columns = [
    "log_area",
    "log_perimeter",
    "compactness",
    "bbox_fill_ratio",
    "centroid_x",
    "centroid_y",
    "floors_feature",
    "floors_known",
    "roof_height_feature",
    "roof_height_known",
    "has_parts_feature"
]

X = (
    model_buildings[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

training_mask = (
    model_buildings["height_source"]
    .eq("overture_height")
    & model_buildings[
        "analysis_height_m"
    ].between(1.5, 100)
)

X_known = X.loc[training_mask]

y_known = model_buildings.loc[
    training_mask,
    "analysis_height_m"
]

print("Available training buildings:", len(X_known))

Available training buildings: 3460


In [33]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X_known,
        y_known,
        test_size=0.20,
        random_state=42
    )
)

height_model = ExtraTreesRegressor(
    n_estimators=300,
    min_samples_leaf=3,
    max_features=0.8,
    n_jobs=-1,
    random_state=42
)

height_model.fit(
    X_train,
    np.log1p(y_train)
)

test_predictions = np.expm1(
    height_model.predict(X_test)
)

mae = mean_absolute_error(
    y_test,
    test_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_predictions
    )
)

r2 = r2_score(
    y_test,
    test_predictions
)

baseline_predictions = np.full(
    len(y_test),
    y_train.median()
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

print("--- HEIGHT MODEL VALIDATION ---")
print(f"Training buildings: {len(X_train):,}")
print(f"Testing buildings: {len(X_test):,}")
print(f"Model MAE: {mae:.2f} m")
print(f"Model RMSE: {rmse:.2f} m")
print(f"Model R²: {r2:.3f}")
print(
    f"Median-baseline MAE: "
    f"{baseline_mae:.2f} m"
)

--- HEIGHT MODEL VALIDATION ---
Training buildings: 2,768
Testing buildings: 692
Model MAE: 1.12 m
Model RMSE: 2.07 m
Model R²: 0.614
Median-baseline MAE: 2.40 m


In [34]:
# Refit using every available direct height
height_model.fit(
    X_known,
    np.log1p(y_known)
)

missing_mask = model_buildings[
    "analysis_height_m"
].isna()

X_missing = X.loc[missing_mask]

predicted_heights = np.expm1(
    height_model.predict(X_missing)
)

predicted_heights = np.clip(
    predicted_heights,
    1.5,
    100
)

# Variation among trees provides uncertainty
individual_predictions = np.vstack([
    np.expm1(
        tree.predict(X_missing.values)
    )
    for tree in height_model.estimators_
])

prediction_uncertainty = (
    individual_predictions.std(axis=0)
)

model_buildings.loc[
    missing_mask,
    "analysis_height_m"
] = predicted_heights

model_buildings.loc[
    missing_mask,
    "height_source"
] = "model_estimate"

model_buildings["height_uncertainty_m"] = np.nan

model_buildings.loc[
    missing_mask,
    "height_uncertainty_m"
] = prediction_uncertainty

model_buildings["height_confidence"] = "provided"

model_buildings.loc[
    model_buildings["height_source"]
    == "estimated_from_floors",
    "height_confidence"
] = "medium"

model_buildings.loc[
    (
        model_buildings["height_source"]
        == "model_estimate"
    )
    & (
        model_buildings[
            "height_uncertainty_m"
        ] <= 2
    ),
    "height_confidence"
] = "medium"

model_buildings.loc[
    (
        model_buildings["height_source"]
        == "model_estimate"
    )
    & (
        model_buildings[
            "height_uncertainty_m"
        ] > 2
    ),
    "height_confidence"
] = "low"

print("--- COMPLETED BUILDING HEIGHTS ---")
print("Total buildings:", len(model_buildings))

print(
    "Remaining missing:",
    model_buildings[
        "analysis_height_m"
    ].isna().sum()
)

print("\nHeight sources:")
print(
    model_buildings[
        "height_source"
    ].value_counts()
)

print("\nConfidence:")
print(
    model_buildings[
        "height_confidence"
    ].value_counts()
)

print("\nFinal height statistics:")
print(
    model_buildings[
        "analysis_height_m"
    ].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

print("\nModel uncertainty:")
print(
    model_buildings.loc[
        model_buildings[
            "height_source"
        ] == "model_estimate",
        "height_uncertainty_m"
    ].describe()
)

--- COMPLETED BUILDING HEIGHTS ---
Total buildings: 3694
Remaining missing: 0

Height sources:
height_source
overture_height          3460
model_estimate            215
estimated_from_floors      19
Name: count, dtype: int64

Confidence:
height_confidence
provided    3460
medium       185
low           49
Name: count, dtype: int64

Final height statistics:
count    3694.000000
mean        6.097181
std         3.933324
min         1.508778
25%         3.391207
50%         5.620733
75%         7.639677
95%        11.263053
99%        20.756998
max        69.510000
Name: analysis_height_m, dtype: float64

Model uncertainty:
count    215.000000
mean       1.372792
std        1.125484
min        0.154886
25%        0.395021
50%        1.226052
75%        1.838365
max        6.932992
Name: height_uncertainty_m, dtype: float64


In [ ]:
final_buildings = model_buildings.to_crs(
    "EPSG:4326"
)

final_height_map = final_buildings.explore(
    column="analysis_height_m",
    cmap="YlOrRd",
    legend=True,
    tooltip=[
        "id",
        "analysis_height_m",
        "height_source",
        "height_confidence",
        "height_uncertainty_m",
        "footprint_area_m2"
    ],
    tiles="CartoDB positron",
    style_kwds={
        "fillOpacity": 0.75,
        "weight": 0.3
    },
    name="Building heights"
)

umd_boundary.explore(
    m=final_height_map,
    color="red",
    fill=False,
    style_kwds={"weight": 3},
    name="UMD boundary"
)

final_height_map

In [36]:
export_columns = [
    "id",
    "overture_height_m",
    "analysis_height_m",
    "height_source",
    "height_confidence",
    "height_uncertainty_m",
    "num_floors",
    "roof_shape",
    "roof_height",
    "has_parts",
    "footprint_area_m2",
    "geometry"
]

final_export = final_buildings[
    export_columns
].copy()

output_path = (
    "data/processed/"
    "umd_buildings_with_heights_final.geojson"
)

os.makedirs(
    "data/processed",
    exist_ok=True
)

final_export.to_file(
    output_path,
    driver="GeoJSON"
)

print("Created:", output_path)

print(
    "File size:",
    round(
        os.path.getsize(output_path)
        / 1_000_000,
        2
    ),
    "MB"
)

Created: data/processed/umd_buildings_with_heights_final.geojson
File size: 2.67 MB


In [37]:
import geopandas as gpd
import os

latest_file = (
    "/content/umd-urban-heat-exposure/data/processed/"
    "umd_buildings_with_heights_final.geojson"
)

check = gpd.read_file(latest_file)

print("Features:", len(check))
print("File size:", round(os.path.getsize(latest_file) / 1_000_000, 2), "MB")
print("Missing heights:", check["analysis_height_m"].isna().sum())

Features: 3694
File size: 2.67 MB
Missing heights: 0


In [38]:
from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

drive_folder = "/content/drive/MyDrive/FortyGuard"
os.makedirs(drive_folder, exist_ok=True)

drive_file = os.path.join(
    drive_folder,
    "umd_buildings_with_heights_final.geojson"
)

shutil.copy2(latest_file, drive_file)

print("Copied to:", drive_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied to: /content/drive/MyDrive/FortyGuard/umd_buildings_with_heights_final.geojson
